In [1]:
import os
import csv
import time
from datetime import datetime, timedelta
import requests
from requests.auth import HTTPBasicAuth
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import itertools
from dotenv import load_dotenv


In [2]:
WORKER_IDS = itertools.count(1)
THREAD_ID_MAP = {}

def get_worker_id():
    tid = threading.get_ident()
    if tid not in THREAD_ID_MAP:
        THREAD_ID_MAP[tid] = next(WORKER_IDS)
    return THREAD_ID_MAP[tid]


In [3]:
load_dotenv()

CITY = os.getenv("CITY")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")
BASE_URL = f"https://{CITY}.pulse.eco/rest"
OUT_DIR = "pulse_data"

print(f'CITY={CITY}')

MAX_WORKERS = 12   
REQUEST_SLEEP = 5 

CITY=bitola


In [4]:
def get_sensors():
    url = f"{BASE_URL}/sensor"
    r = requests.get(url, auth=HTTPBasicAuth(USERNAME, PASSWORD))
    if r.status_code != 200:
        raise Exception("Failed to fetch sensors:", r.text)
    return r.json()

valid_statuses = {
    "ACTIVE",
    "ACTIVE_UNCONFIRMED",
    "NOT_CLAIMED",
    "NOT_CLAIMED_UNCONFIRMED"
}


sensors = get_sensors()
filtered = [s for s in sensors if s["status"] in valid_statuses]

print("Total sensors:", len(sensors))
print("Filtered sensors:", len(filtered))
print("Example sensor:", filtered[0])

Total sensors: 22
Filtered sensors: 22
Example sensor: {'sensorId': 'd241a044-0a06-40c2-9d90-c91fd0a95060', 'position': '40.99851020119874,21.243798033728062', 'comments': 'Postaveno od RC Bitola na 19.08.2024', 'type': '3', 'description': 'RC Bitola Nize Pole', 'status': 'ACTIVE'}


In [5]:

def parse_latlon(pos):
    try:
        lat, lon = pos.split(",")
        return lat.strip(), lon.strip()
    except:
        return None, None

def save_week_csv(filepath, data):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    with open(filepath, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["timestamp", "sensorId", "lat", "lon", "type", "value"])

        for entry in data:
            lat, lon = parse_latlon(entry.get("position", ""))
            writer.writerow([
                entry["stamp"],
                entry["sensorId"],
                lat,
                lon,
                entry["type"],
                entry["value"]
            ])

request_count = 0
start_time_global = time.time()
thread_lock = threading.Lock()

def fetch_raw(sensor_id, start, end):
    global request_count
    worker_id = get_worker_id()

    url = f"{BASE_URL}/dataRaw"
    params = {
        "sensorId": sensor_id,
        "from": start.isoformat() + "Z",
        "to": end.isoformat() + "Z"
    }

    r = requests.get(url, params=params, auth=HTTPBasicAuth(USERNAME, PASSWORD))

    with thread_lock:
        request_count += 1
        elapsed = time.time() - start_time_global
        print(f"[Worker {worker_id}] Request #{request_count} | Elapsed: {elapsed:.1f}s | Status: {r.status_code}")

    time.sleep(5)

    if r.status_code == 429:
        print(f"[Worker {worker_id}] Rate limited — waiting 60 seconds…")
        time.sleep(60)
        return fetch_raw(sensor_id, start, end)

    if r.status_code != 200:
        print(f"[Worker {worker_id}] Error {r.status_code}: {r.text}")
        return []

    return r.json()



In [7]:
def download_range_for_sensor(sensor_id, start_date, end_date):
    worker_id = get_worker_id()
    print(f"[Worker {worker_id}] Starting sensor {sensor_id}")

    total_days = (end_date - start_date).days
    total_weeks = total_days // 7 + 1

    current = start_date
    current_year = current.year
    week_counter = 1

    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)

        # reset week counter on new year
        if current.year != current_year:
            current_year = current.year
            week_counter = 1

        print(f"[Worker {worker_id}] Sensor {sensor_id} | "
              f"Week {week_counter}/{total_weeks} | {current.date()} → {week_end.date()}")

        folder = f"{OUT_DIR}/{current_year}/week_{week_counter}"
        os.makedirs(folder, exist_ok=True)

        filename = f"{sensor_id}_{current.date()}_{week_end.date()}.csv"
        filepath = f"{folder}/{filename}"

        if os.path.exists(filepath):
            print(f"[Worker {worker_id}] Skipping existing: {filepath}")
            current = week_end
            week_counter += 1
            continue

        data = fetch_raw(sensor_id, current, week_end)

        save_week_csv(filepath, data)

        current = week_end
        week_counter += 1

    print(f"[Worker {worker_id}] Finished sensor {sensor_id}")
    return sensor_id



def download_all_sensors_parallel(sensors, start_date, end_date):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_range_for_sensor, s["sensorId"], start_date, end_date): s
            for s in sensors
        }

        for future in as_completed(futures):
            sensor = futures[future]
            try:
                sid = future.result()
                print(f"[DONE] Sensor {sid}")
            except Exception as e:
                print(f"[ERROR] {sensor['sensorId']}: {e}")



In [ ]:
start_date = datetime(2025, 12, 1, 0, 0, 0)
end_date   = datetime(2026, 3, 2, 0, 0, 0)

download_all_sensors_parallel(filtered, start_date, end_date)

[Worker 1] Starting sensor d241a044-0a06-40c2-9d90-c91fd0a95060
[Worker 1] Sensor d241a044-0a06-40c2-9d90-c91fd0a95060 | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 2] Starting sensor fec52a19-9148-4350-a1b4-ae0da05ee199
[Worker 2] Sensor fec52a19-9148-4350-a1b4-ae0da05ee199 | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 3] Starting sensor be427cee-4c3a-4aa2-a1ce-9795a74533be
[Worker 3] Sensor be427cee-4c3a-4aa2-a1ce-9795a74533be | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 4] Starting sensor c3f3da9b-9fd3-4037-94d3-598d655e6be9
[Worker 4] Sensor c3f3da9b-9fd3-4037-94d3-598d655e6be9 | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 5] Starting sensor d7060523-b163-4cc3-bc94-970dac72a38a
[Worker 5] Sensor d7060523-b163-4cc3-bc94-970dac72a38a | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 6] Starting sensor 87f82783-853b-417d-8964-b5cf11e44873
[Worker 6] Sensor 87f82783-853b-417d-8964-b5cf11e44873 | Week 1/14 | 2025-12-01 → 2025-12-08
[Worker 7] Starting sensor e20e9778-a020-4b86-932a-b7ab6a7